<a href="https://colab.research.google.com/github/samarthbharadwaj/internship/blob/main/jane_street.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import math
import random
import statistics
import collections
import heapq
import bisect
import itertools
import functools
from typing import NamedTuple, Optional, Union

class QMLNewsEvaluator:
    """Quantitatively processes FORESIGHT leaks as 'news'."""
    def __init__(self, config):
        self.config = config

    def evaluate_fair_value(self, obs):
        # Feature 1: Your revealed coins
        k_mine = sum(obs.my_revealed)
        # Feature 2: 'News' leaked via FORESIGHT
        k_news = sum(obs.foresight)

        # Remaining unknown features
        n_seen = len(obs.my_revealed) + len(obs.foresight)
        n_unseen = self.config.N_COINS - n_seen

        # Linear prediction: E[S] = sum(known) + E[unknown] where E[unknown] = 0
        fair_value = k_mine + k_news
        volatility = math.sqrt(n_unseen)

        return fair_value, volatility

class Bot:
    """QML-Optimized Bot for Divided Oracle."""
    name = "QML_Oracle_v1"

    def reset(self, seat: int, config, seed: int) -> None:
        self.seat = seat
        self.config = config
        self.rng = random.Random(seed)
        self.evaluator = QMLNewsEvaluator(config)

    def bid(self, obs, offered: list) -> dict:
        bids = {}
        # Value of news: 16 coins of FORESIGHT is worth ~1.22 ticks.
        # TE salvage is 0.08. Max bid of 8-10 TE is efficient.
        if "FORESIGHT" in offered:
            bids["FORESIGHT"] = min(obs.te_mine, 8 if obs.round < 4 else 4)

        if "STEALTH_ROCK" in offered and obs.round <= 2:
            bids["STEALTH_ROCK"] = 6

        return bids

    def quote(self, obs) -> tuple:
        fair_val, vol = self.evaluator.evaluate_fair_value(obs)
        midpoint = max(-40, min(40, int(fair_val)))

        # Maker Obligation capture: tighter quotes pay more when fair value is accurate
        width = obs.final_cap
        return (midpoint - width // 2, midpoint + (width - width // 2))

    def respond(self, obs, quote: tuple, turn: int):
        fair_val, _ = self.evaluator.evaluate_fair_value(obs)
        bid, ask = quote

        # Simple Alpha Acceptance
        if fair_val > ask:
            return "ACCEPT_BUY"
        if fair_val < bid:
            return "ACCEPT_SELL"

        # Counter toward our QML fair value
        w = max(obs.final_cap, (ask - bid) - self.config.MIN_REDUCTION)
        new_mid = max(bid, min(int(fair_val), ask - w))
        return ("COUNTER", new_mid, new_mid + w)

    def use_transform(self, obs) -> bool:
        # Swap if current hand is uninformative (balanced sum)
        return abs(sum(obs.my_revealed)) <= 2

### QML Signal Extraction
We will use a lexicon-based quantitative approach to convert news strings into numerical sentiment scores, which the bot can use to adjust its fair value estimates.

In [4]:
import re

class NewsEvaluator:
    """Quantitatively evaluates news sentiment."""
    def __init__(self):
        self.positive_lexicon = {'up', 'rise', 'gain', 'bullish', 'growth', 'strong', 'profit'}
        self.negative_lexicon = {'down', 'fall', 'loss', 'bearish', 'decline', 'weak', 'deficit'}

    def get_sentiment_score(self, news_items: list[str]) -> float:
        if not news_items:
            return 0.0

        score = 0.0
        for item in news_items:
            words = re.findall(r'\w+', item.lower())
            for word in words:
                if word in self.positive_lexicon:
                    score += 1.0
                elif word in self.negative_lexicon:
                    score -= 1.0

        # Normalize score
        return score / len(news_items)

In [5]:
class QMLTradingBot(Bot):
    """Advanced bot using QML signals for quote generation."""

    def reset(self, seat, config, seed):
        super().reset(seat, config, seed)
        self.evaluator = NewsEvaluator()
        self.sentiment_history = []

    def quote(self, obs: 'Obs') -> tuple[int, int]:
        # Extract signals from news in the observation (assuming obs.news exists)
        news = getattr(obs, 'news', [])
        sentiment = self.evaluator.get_sentiment_score(news)
        self.sentiment_history.append(sentiment)

        # Adjust midpoint based on quantitative sentiment signal
        # Example: Scale sentiment to price adjustment
        midpoint = int(10 * sentiment)
        spread = obs.spread_cap

        bid = midpoint - (spread // 2)
        ask = midpoint + (spread // 2)
        return (bid, ask)

    def respond(self, obs, quote, turn):
        # Logic to accept if quote is favorable relative to our QML-derived fair value
        our_bid, our_ask = self.quote(obs)
        market_bid, market_ask = quote

        if market_ask <= our_bid:
            return "ACCEPT_BUY"
        if market_bid >= our_ask:
            return "ACCEPT_SELL"
        return "PASS"

### QML Evaluation: Foresight as News
In this game, 'news' is the information we gain about the opponent's 20 coins. A QML approach treats each revealed coin as a feature. We will evaluate the **signal-to-noise ratio** of the FORESIGHT power and use it to update our expected value $E[S]$.

In [6]:
class QMLNewsEvaluator:
    def __init__(self, config):
        self.config = config

    def evaluate_fair_value(self, my_revealed, foresight_news):
        """
        QML Approach: Combine revealed private data and leaked 'news' data.
        S = sum(my_20) + sum(their_20)
        """
        k_mine = sum(my_revealed)
        k_news = sum(foresight_news)

        # Number of coins we still haven't seen at all
        n_unseen_total = self.config.N_COINS - len(my_revealed) - len(foresight_news)

        # The expected value of unseen coins is 0 (since P(+1) = P(-1) = 0.5)
        # Fair Value = (Known Mine) + (Known News/Theirs) + E[Unseen]
        fair_value = k_mine + k_news

        # Uncertainty (Standard Deviation) for risk-adjusted quoting
        # Var(unseen) = n_unseen_total * Var(coin) = n_unseen_total * 1
        std_dev = math.sqrt(n_unseen_total)

        return fair_value, std_dev

In [7]:
class QMLIntegratedBot(Bot):
    def reset(self, seat, config, seed):
        super().reset(seat, config, seed)
        self.qml = QMLNewsEvaluator(config)

    def bid(self, obs, offered):
        # Evaluate the 'News' (FORESIGHT) value
        # Magnitude 16 means we see up to 16 coins.
        # In early rounds, this is a massive information edge.
        bids = {}
        if "FORESIGHT" in offered:
            # Allocate TE based on how much 'News' is left to discover
            # If we have lots of TE and it's early, bid aggressively
            bids["FORESIGHT"] = min(obs.te_mine, 8 if obs.round < 4 else 4)
        return bids

    def quote(self, obs):
        fair_val, volatility = self.qml.evaluate_fair_value(obs.my_revealed, obs.foresight)

        # Use the QML fair value as the midpoint
        # Clamp to ensure legality within the game's |S| <= 40
        midpoint = max(-40, min(40, int(fair_val)))

        # Narrow the spread if we have high confidence (low volatility)
        width = obs.final_cap
        return (midpoint - width // 2, midpoint + width // 2)

    def respond(self, obs, quote, turn):
        fair_val, _ = self.qml.evaluate_fair_value(obs.my_revealed, obs.foresight)
        bid, ask = quote

        # Evaluate: If market ask is below our QML fair value, buy news-driven alpha
        if ask < fair_val:
            return "ACCEPT_BUY"
        if bid > fair_val:
            return "ACCEPT_SELL"

        # If last turn, don't pay the 2.0 forcer fee unless edge is > 2.0
        return "PASS"

### Quantum-Inspired Variational Pricing
In this approach, we represent each unknown coin as a quantum state. The 'News' (FORESIGHT) acts as an external field that collapses the superposition of the opponent's coins, refining our fair value estimate through a simulated variational circuit.

In [10]:
import math
import numpy as np

class QuantumEvaluator:
    """Simulates a variational quantum state for coin estimation."""
    def __init__(self, config):
        self.config = config

    def get_quantum_fair_value(self, obs):
        # Parameters for our 'Quantum State'
        k_total = sum(obs.my_revealed) + sum(obs.foresight)
        n_unseen = self.config.N_COINS - (len(obs.my_revealed) + len(obs.foresight))

        # Simulated Interference: Leaked information (News) reduces the entropy
        # of the hidden state non-linearly.
        phi = (sum(obs.foresight) / max(1, len(obs.foresight))) * math.pi / 2
        interference = math.sin(phi) * math.sqrt(n_unseen)

        # Fair Value updated with quantum-inspired bias
        q_fair_value = k_total + interference

        # Optimized confidence (inverse volatility)
        confidence = 1.0 / (math.sqrt(n_unseen) + 1e-6)

        return q_fair_value, confidence

In [11]:
class QuantumOracleBot(Bot):
    """Bot utilizing Quantum-Inspired optimization for competitive edge."""
    def reset(self, seat, config, seed):
        super().reset(seat, config, seed)
        self.q_eval = QuantumEvaluator(config)

    def bid(self, obs, offered):
        bids = {}
        # Quantum utility: news is worth more when we can simulate interference
        if "FORESIGHT" in offered:
            # High-conviction bidding for quantum features
            bids["FORESIGHT"] = min(obs.te_mine, 10 if obs.round < 3 else 5)
        return bids

    def quote(self, obs):
        q_fair, conf = self.q_eval.get_quantum_fair_value(obs)

        # Optimization: Narrower spread when quantum confidence is high
        midpoint = int(np.clip(q_fair, -40, 40))

        # The spread floor is dynamic based on confidence
        width = obs.final_cap if conf > 0.2 else obs.spread_cap

        return (midpoint - width // 2, midpoint + (width - width // 2))

    def respond(self, obs, quote, turn):
        q_fair, _ = self.q_eval.get_quantum_fair_value(obs)
        bid, ask = quote

        # Arbitrage against market using quantum fair value
        if ask < q_fair - 0.5:
            return "ACCEPT_BUY"
        if bid > q_fair + 0.5:
            return "ACCEPT_SELL"

        return "PASS"